# Priprema za 1. kolokvij

**I051 Računalno jezikoslovlje** — teme P2–P7 i vježbe V2–V7.

Kolokvij ima **5 zadataka**. Svaki zadatak ima:

* **teorijsko pitanje** — kratak pisani odgovor (pitanja su iz popisa `kolokvij_1_pitanja.docx`),
* **praktični zadatak** — kratak program u Pythonu, kao na vježbama.

Podaci su zadani u samoj bilježnici, pa nije potrebna nijedna datoteka. Potrebni paketi: `numpy` i `nltk` (isti kao na vježbama).

U ovoj verziji uz svaki zadatak su i **rješenja**.

In [1]:
import math
import re
from collections import Counter

import numpy as np

---
## Zadatak 1. Riječi i tokeni (P2, V2)

**Teorijsko pitanje.** Objasnite razliku između **različnice** i **pojavnice**. Što kaže **Heapsov zakon** i zašto tekst na hrvatskom ima više različnica od engleskog teksta iste duljine?

**Odgovor.** Pojavnica je svako pojavljivanje riječi ...

**Praktični zadatak.**

1. Napišite funkciju `clean(text)` koja vraća listu riječi teksta: samo riječi od slova, malim slovima. Za zadani tekst izračunajte $N$, $|V|$ i pet najčešćih riječi.
2. **BPE.** Riječi malog korpusa s predavanja rastavljene su na znakove. Napišite funkciju `count_pairs(words)` koja broji parove susjednih simbola (unutar riječi) i pomoću zadane funkcije `merge` napravite **tri spajanja** algoritma BPE. Nakon svakog spajanja ispišite spojeni par i novi rastav riječi.

In [2]:
text = ("Na stolu je knjiga, na knjizi je olovka, a na olovki je mrav. "
        "Knjiga je stara, a olovka je nova. Mrav ne čita knjigu.")

def clean(text):
    return [w.lower() for w in re.findall(r"\w+", text) if w.isalpha()]

words = clean(text)
print("N =", len(words), "  |V| =", len(set(words)))
print(Counter(words).most_common(5))

N = 24   |V| = 14
[('je', 5), ('na', 3), ('knjiga', 2), ('olovka', 2), ('a', 2)]


In [3]:
corpus = [list(w) for w in "set new new renew reset renew".split()]   # riječ = lista simbola

def count_pairs(words):
    pairs = Counter()
    for w in words:
        for a, b in zip(w, w[1:]):        # susjedni simboli unutar riječi
            pairs[(a, b)] += 1
    return pairs

def merge(words, pair):
    # zadano: svako pojavljivanje para (a, b) zamijeni jednim simbolom a + b
    out = []
    for w in words:
        new, i = [], 0
        while i < len(w):
            if i + 1 < len(w) and (w[i], w[i + 1]) == pair:
                new.append(w[i] + w[i + 1]); i += 2
            else:
                new.append(w[i]); i += 1
        out.append(new)
    return out

for step in range(3):
    pair, n = count_pairs(corpus).most_common(1)[0]
    corpus = merge(corpus, pair)
    print(step + 1, pair, n, "→", [" ".join(w) for w in corpus])

1 ('n', 'e') 4 → ['s e t', 'ne w', 'ne w', 'r e ne w', 'r e s e t', 'r e ne w']
2 ('ne', 'w') 4 → ['s e t', 'new', 'new', 'r e new', 'r e s e t', 'r e new']
3 ('r', 'e') 3 → ['s e t', 'new', 'new', 're new', 're s e t', 're new']


Parovi *n e* i *e w* oba se pojavljuju 4 puta; `most_common` tada vraća par koji je prvi prebrojen. Prva dva spajanja (*n + e*, pa *ne + w*) ista su kao na predavanju. Treće je *r + e*: na predavanju je prvo spojen razmak s *r* (▁r), jer je korpus imao oznaku ▁ za početak riječi.

---
## Zadatak 2. Regularni izrazi i udaljenost uređivanja (P3–P4, V3–V4)

**Teorijsko pitanje.** Definirajte **Levenshteinovu udaljenost**. Zašto se računa **dinamičkim programiranjem**? Što je $D[i, j]$, koji su početni uvjeti i kako glasi rekurzivna relacija?

**Odgovor.** Levenshteinova udaljenost je ...

**Praktični zadatak.**

1. Iz zadanog teksta regularnim izrazom izdvojite sve datume u obliku `2026-10-15` i funkcijom `re.sub` s grupama za hvatanje pretvorite ih u hrvatski zapis `15. 10. 2026.`
2. Za svaku pogrešno napisanu riječ predložite najbližu riječ iz zadanog rječnika (funkcija `edit_distance` iz NLTK-a). Kad je više riječi jednako blizu, uzmite češću.

In [4]:
news = ("Sjednica je održana 2026-10-15 u Zagrebu, a sljedeća je zakazana za 2026-11-12 u Osijeku. "
        "Rok za prijave bio je 2026-09-30 u podne.")

print(re.findall(r"\d{4}-\d{2}-\d{2}", news))
print(re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3. \2. \1.", news))

['2026-10-15', '2026-11-12', '2026-09-30']
Sjednica je održana 15. 10. 2026. u Zagrebu, a sljedeća je zakazana za 12. 11. 2026. u Osijeku. Rok za prijave bio je 30. 09. 2026. u podne.


In [5]:
from nltk import edit_distance

freq = Counter({"jezik": 120, "je": 5000, "jež": 3, "predsjednik": 300, "predsjednica": 40,
                "sveučilište": 80, "sveučilišta": 60, "hrvatski": 400, "hrvatska": 900})

def suggest(word):
    return min(freq, key=lambda w: (edit_distance(word, w), -freq[w]))   # najbliža, pa češća

for word in ["jezk", "predsjedik", "sveučilšte", "hrvatksi"]:
    best = suggest(word)
    print(f"{word:12} → {best:12} (udaljenost {edit_distance(word, best)})")

jezk         → jezik        (udaljenost 1)
predsjedik   → predsjednik  (udaljenost 1)
sveučilšte   → sveučilište  (udaljenost 1)
hrvatksi     → hrvatski     (udaljenost 2)


Za *hrvatksi* (zamijenjena dva susjedna slova) udaljenost je 2: Levenshteinova udaljenost zamjenu susjednih slova broji kao dvije operacije.

---
## Zadatak 3. N-gramski jezični modeli (P5, V5)

**Teorijsko pitanje.** Definirajte **perpleksiju** i objasnite zašto je manja perpleksija bolja. Zašto bigramski model procijenjen bez izglađivanja (MLE) može na testnom skupu imati beskonačnu perpleksiju i kako to rješava **Laplaceovo izglađivanje**?

**Odgovor.** Perpleksija testnog skupa ...

**Praktični zadatak.** Zadan je mali korpus s predavanja.

1. Prebrojite unigrame i bigrame (rečenicama dodajte `<s>` i `</s>`) i izračunajte MLE vjerojatnosti $P(\text{sam} \mid \text{ja})$ i $P(\text{Ana} \mid \text{<s>})$.
2. Napišite `p_laplace(word, prev)` i izračunajte perpleksiju testne rečenice *Ana sam ja* s MLE-om i s Laplaceovim izglađivanjem. Vjerojatnost računajte zbrajanjem logaritama.

In [6]:
corpus = ["ja sam Ana", "Ana ja sam", "ja ne volim zelena jaja i šunku"]

unigrams, bigrams = Counter(), Counter()
for sentence in corpus:
    words = ["<s>"] + sentence.split() + ["</s>"]
    unigrams.update(words)
    bigrams.update(zip(words, words[1:]))

def p_mle(word, prev):
    return bigrams[(prev, word)] / unigrams[prev]

print("P(sam | ja)  =", round(p_mle("sam", "ja"), 3))
print("P(Ana | <s>) =", round(p_mle("Ana", "<s>"), 3))

P(sam | ja)  = 0.667
P(Ana | <s>) = 0.333


In [7]:
V = len(unigrams)                     # veličina rječnika (s <s> i </s>)

def p_laplace(word, prev):
    return (bigrams[(prev, word)] + 1) / (unigrams[prev] + V)

def perplexity(sentence, p):
    words = ["<s>"] + sentence.split() + ["</s>"]
    log_prob = 0
    for prev, word in zip(words, words[1:]):
        prob = p(word, prev)
        if prob == 0:
            return math.inf           # neviđeni bigram: vjerojatnost 0
        log_prob += math.log(prob)
    n = len(words) - 1                # broj predviđanja
    return math.exp(-log_prob / n)

test = "Ana sam ja"
print("V =", V)
print("MLE:    ", perplexity(test, p_mle))
print("Laplace:", round(perplexity(test, p_laplace), 2))

V = 11
MLE:     inf
Laplace: 11.34


Bigram *Ana sam* nikad se ne pojavljuje u korpusu, pa MLE daje beskonačnu perpleksiju; s Laplaceom je perpleksija konačna.

---
## Zadatak 4. Klasifikacija teksta i logistička regresija (P6, V6)

**Teorijsko pitanje.** Kako glasi **unakrsna entropija** za binarnu logističku regresiju? Objasnite **gradijentni spust**, ulogu stope učenja $\eta$ i protumačite gradijent $(\hat y - y)\,x_j$.

**Odgovor.** Model daje ...

**Praktični zadatak.**

1. Napišite `sigmoid(z)` i `predict(w, b, x)` te napravite **jedan korak** gradijentnog spusta za primjer s predavanja: $\mathbf{x} = [1, 0, 0]$, $y = 1$, početne težine i pomak 0, $\eta = 0{,}1$. Ispišite gubitak prije i poslije koraka.
2. Klasifikator je 50 rečenica pripisao SETimesu, od kojih je 40 točno; u testu je 80 rečenica iz SETimesa. Iz zadanih lista zlatnih oznaka i odluka izračunajte matricu zabune, preciznost, odziv i F1 za razred `set`.

In [8]:
def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def predict(w, b, x):
    return sigmoid(sum(wi * xi for wi, xi in zip(w, x)) + b)

def loss(w, b, x, y):
    y_hat = predict(w, b, x)
    return -math.log(y_hat) if y == 1 else -math.log(1 - y_hat)

w, b, x, y, lr = [0, 0, 0], 0, [1, 0, 0], 1, 0.1
print("prije:", w, b, "ŷ =", predict(w, b, x), "gubitak =", round(loss(w, b, x, y), 4))

y_hat = predict(w, b, x)
w = [wi - lr * (y_hat - y) * xi for wi, xi in zip(w, x)]
b = b - lr * (y_hat - y)
print("poslije:", w, b, "ŷ =", round(predict(w, b, x), 4), "gubitak =", round(loss(w, b, x, y), 4))

prije: [0, 0, 0] 0 ŷ = 0.5 gubitak = 0.6931
poslije: [0.05, 0.0, 0.0] 0.05 ŷ = 0.525 gubitak = 0.6444


In [9]:
gold = ["set"] * 40 + ["hr"] * 10 + ["set"] * 40 + ["hr"] * 30
pred = ["set"] * 40 + ["set"] * 10 + ["hr"] * 40 + ["hr"] * 30

counts = Counter(zip(gold, pred))      # (zlatna, sustav) -> broj
tp, fp = counts[("set", "set")], counts[("hr", "set")]
fn, tn = counts[("set", "hr")], counts[("hr", "hr")]
print("tp, fp, fn, tn =", tp, fp, fn, tn)

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
accuracy = (tp + tn) / len(gold)
print(f"P = {precision:.3f}, R = {recall:.3f}, F1 = {f1:.3f}, točnost = {accuracy:.3f}")

tp, fp, fn, tn = 40 10 40 30
P = 0.800, R = 0.500, F1 = 0.615, točnost = 0.583


Preciznost je visoka (0,8), ali odziv je samo 0,5: klasifikator pronalazi tek polovicu rečenica iz SETimesa. F1 kao harmonijska sredina to kažnjava (0,615).

---
## Zadatak 5. Vektorska semantika (P7, V7)

**Teorijsko pitanje.** Zašto za sličnost vektora riječi koristimo **kosinus**, a ne skalarni umnožak? Zašto česte riječi kvare vektore iz brojanja? Definirajte **PMI** i **PPMI** i protumačite ih.

**Odgovor.** Skalarni umnožak raste...

**Praktični zadatak.** Zadana je matrica riječ–kontekst s predavanja (korpus SET, prozor ±4, šest stupaca).

1. Napišite funkciju `cosine(v, w)` i izračunajte kosinus za parove *kuna–eura* i *kuna–ponedjeljak*.
2. Napišite funkciju `ppmi(M)` po formuli s predavanja (vjerojatnosti iz zbrojeva retka, stupca i cijele matrice) i ispišite matricu PPMI zaokruženu na dvije decimale. Koji kontekst ima najveću težinu za *ponedjeljak* i zašto?

In [10]:
words = ["kuna", "eura", "ponedjeljak", "utorak"]
contexts = ["milijuna", "milijardi", "oko", "izjavio", "siječnja", "("]
M = np.array([[62, 22, 11, 0, 0, 6],
              [67, 27, 25, 0, 0, 4],
              [0, 0, 0, 10, 7, 40],
              [0, 0, 0, 6, 6, 28]], dtype=float)

def cosine(v, w):
    return v @ w / (np.linalg.norm(v) * np.linalg.norm(w))

print("kuna – eura:       ", round(cosine(M[0], M[1]), 3))
print("kuna – ponedjeljak:", round(cosine(M[0], M[2]), 3))

kuna – eura:        0.985
kuna – ponedjeljak: 0.086


In [11]:
def ppmi(M):
    N = M.sum()
    p_wc = M / N
    p_w = M.sum(axis=1, keepdims=True) / N      # zbrojevi redaka
    p_c = M.sum(axis=0, keepdims=True) / N      # zbrojevi stupaca
    with np.errstate(divide="ignore"):
        pmi = np.log2(p_wc / (p_w * p_c))       # log2(0) = -inf
    return np.maximum(pmi, 0)

P = ppmi(M)
print(" " * 12, " ".join(f"{c:>9}" for c in contexts))
for word, row in zip(words, P):
    print(f"{word:12}", " ".join(f"{v:9.2f}" for v in row))

              milijuna milijardi       oko   izjavio  siječnja         (
kuna              0.61      0.51      0.00      0.00      0.00      0.00
eura              0.44      0.52      0.86      0.00      0.00      0.00
ponedjeljak       0.00      0.00      0.00      1.82      1.60      1.53
utorak            0.00      0.00      0.00      1.59      1.89      1.53


Za *ponedjeljak* najveću težinu ima kontekst *izjavio* (1,82), a ne »(« (1,53), iako je zagrada s 40 pojavljivanja daleko najčešći kontekst. Zagrada se pojavljuje i uz *kunu* i *euro*, pa je manje specifična za dane; *izjavio* se u ovoj matrici pojavljuje samo uz dane. PPMI dakle ne nagrađuje učestalost, nego **neobičnu** učestalost uz baš tu riječ. Primijetite i da su sve vrijednosti »nemogućih« parova (npr. *kuna* – *izjavio*) jednake 0.